## Импорты и загрузка конфига

In [1]:
%pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML
%pip install --upgrade tiktoken protobuf sentencepiece
%pip install accelerate>=1.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 7.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.


In [1]:
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

In [3]:
if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

Cloning into 'text_processing_final_project'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 60 (delta 25), reused 52 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 19.46 KiB | 9.73 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/text_processing_final_project


In [ ]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
import importlib

import warnings
warnings.filterwarnings("ignore")

config = load_config("config.yaml")

print("Config loaded:")
print(config)

Config loaded:
{'dataset_source': 'tapaco', 'source_params': {'tapaco': {'language': 'ru', 'pair_mode': 'all_pairs'}, 'backtranslation': {'input_file': 'data/sentences.txt', 'from_lang': 'ru', 'to_lang': 'en', 'api': 'googletrans'}, 'custom_merge': {'files': ['data/ds1.csv', 'data/ds2.csv']}}, 'preprocessing': {'steps': ['remove_duplicates', 'normalize_text', 'filter_trivial_pairs', 'filter_by_length', 'filter_edit_distance', 'filter_semantic_similarity'], 'remove_duplicates': {}, 'normalize_text': {'lower': True, 'trim': True, 'remove_brackets': True}, 'filter_trivial_pairs': {'min_edit_ratio': 0.18, 'min_len': 6, 'max_len': 50}, 'filter_by_length': {'max_tokens': 128, 'tokenizer': 'cointegrated/rut5-base'}, 'filter_edit_distance': {'min_edit_ratio': 0.1, 'max_edit_ratio': 0.9}, 'filter_semantic_similarity': {'threshold': 0.8, 'model': 'sentence-transformers/LaBSE', 'batch_size': 32}}, 'model_name': 'rut5_paraphraser', 'model_config': {'pretrained_name': 'cointegrated/rut5-base', 'max

## Получение и обработка датасета

In [ ]:
source = config["dataset_source"]
source_params = config["source_params"].get(source, {})
module = importlib.import_module(f"dataset.create_dataset.{source}")
df = getattr(module, "load_or_create")(source_params)
#df = df.sample(n=len(df)//2, random_state=42)
df = df.sample(n=1000, random_state=42)

print(f"Dataset size: {len(df)}")
df.head()

Dataset size: 408750


,original,paraphrase
6507,Он стал смеяться.,Я стал плакать.
263326,Погода облачная.,Начинается ночь.
646969,У меня нет абсолютно никаких проблем.,У меня вообще нет никаких проблем.
124858,Он худой.,Он поджарый.
126185,Ты чего вернулся?,Ты зачем вернулся?


In [4]:
from dataset.prepare_dataset.prepare import prepare_dataset
df_clean = prepare_dataset(df, config["preprocessing"])
print(f"Clean dataset size: {len(df_clean)}")

Running filter_semantic_similarity: 100%|██████████| 6/6 [11:32<00:00, 115.43s/step]

Clean dataset size: 35081


In [5]:
train_loader, val_loader = build_dataloaders(df_clean, config["model_config"])
print(f"Train samples: {len(train_loader.dataset)}, Val samples: {len(val_loader.dataset)}")

Train samples: 31572, Val samples: 3509


## Создание модели и обучение

In [8]:
from models.rut5_paraphraser.model import ParaphraserModel
from models.trainer import train_model

model = ParaphraserModel(config["model_config"], device="cuda" if __import__('torch').cuda.is_available() else "cpu")
trained_model = train_model(
    model,
    train_loader,
    val_loader,
    config["model_config"],
    config["trainer_config"],
    config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

model.safetensors:   0%|          | 0.00/977M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/31572 [00:00<?, ? examples/s]

Map:   0%|          | 0/3509 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed. Model saved to models/rut5_paraphraser/saves


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    !cp -r "models/rut5_paraphraser/saves" "/content/drive/MyDrive/colab_export/"

    from google.colab import runtime
    runtime.unassign()


Mounted at /content/drive


## Оценка доп. метриками

In [6]:
 
from models.model_use import load_model, evaluate_model

model = load_model(
    config["model_name"],
    config["model_config"],
    checkpoint_path=config["trainer_config"]["output_dir"]
)

test_metrics = evaluate_model(model, val_loader, config["metrics_config"])
print("Test metrics:", test_metrics)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 7826.13it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 7614.72it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 9045.12it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/ruber

Test metrics: {'bleu': 16.81179269521362, 'bertscore': np.float64(0.7746533758677226), 'cosine_similarity': np.float32(0.85322255)}
